# SIMON / D2-ROCm — czy dryf jest *vendorowy*?

**Pytanie:** czy uczciwy node na AMD policzy to samo, co uczciwy node na NVIDIA?
Jesli nie, naiwne porownanie odciskow **karze uczciwych uczestnikow**.

**Metoda:** dokladnie to samo ramie A, ktore zmierzylismy na CUDA —
`Qwen2.5-1.5B-Instruct`, fp16, transformers, batch=1, te same `input_ids`.
Obserwabla: logprob faktycznego nastepnego tokenu na kazdej pozycji promptu.

**Uwaga o pulapce:** poprzedni pomiar miedzysprzetowy (D1) byl obciazony
confoundem. Dlatego tu mierzymy **dwie implementacje uwagi** (`eager` i `sdpa`)
i zapisujemy wersje bibliotek — zeby roznica "AMD vs NVIDIA" nie okazala sie
roznica "sdpa vs eager".

**Higiena sesji:** limit 3 h/dobe, liczy sie czas istnienia poda.
Wyniki ida do pamieci trwalej, model tez — inaczej wyparuja.

In [ ]:
# 1. Co to za maszyna (idzie do rekordu pomiaru)
import subprocess, sys, json, platform
import torch

srodowisko = {
    'torch': torch.__version__,
    'hip': getattr(torch.version, 'hip', None),
    'cuda': getattr(torch.version, 'cuda', None),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'python': platform.python_version(),
}
assert torch.cuda.is_available(), 'brak GPU — sesja bez akceleratora, nie ma czego mierzyc'
assert srodowisko['hip'], 'to NIE jest build ROCm (torch.version.hip pusty) — zly notebook/sesja'
print(json.dumps(srodowisko, indent=2))

In [ ]:
# 2. Pamiec trwala: /persistent albo /workspace — zalezy od URL-a sesji
import os, pathlib

TRWALE = next((p for p in ('/persistent', '/workspace') if os.path.isdir(p)), None)
assert TRWALE, 'nie znalazlem /persistent ani /workspace — wyniki NIE przetrwaja sesji'
KAT = pathlib.Path(TRWALE) / 'simon-d2'
KAT.mkdir(exist_ok=True)
os.environ['HF_HOME'] = str(pathlib.Path(TRWALE) / 'hf')   # model tez trwaly, ~3 GB
print('wyniki ->', KAT)
print('model  ->', os.environ['HF_HOME'])

try:
    import transformers
except ImportError:
    !pip -q install transformers
    import transformers
srodowisko['transformers'] = transformers.__version__
print('transformers', transformers.__version__)

In [ ]:
# 3. Te same input_ids, co na CUDA — tokenizacja nie moze byc zmienna
import hashlib
from transformers import AutoTokenizer

MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
TEKST = ('Weryfikacja obliczen w sieci rozproszonej wymaga, zeby wykonawca zobowiazal sie '
         'do sladu wykonania, zanim pozna punkty kontroli. Inaczej moze dobrac, co pokaze. ' * 4)

tok = AutoTokenizer.from_pretrained(MODEL)
IDS = tok(TEKST)['input_ids']
ODCISK_IDS = hashlib.sha256(json.dumps(IDS).encode()).hexdigest()[:16]
print(f'tokenow: {len(IDS)}  (na CUDA bylo 229)')
print('odcisk input_ids:', ODCISK_IDS)
assert len(IDS) == 229, f'INNA TOKENIZACJA ({len(IDS)} != 229) — porownanie byloby bez sensu'

In [ ]:
# 4. JEDNA sztuka najpierw — sprawdzam, czy w ogole liczy sensownie
from transformers import AutoModelForCausalLM

def zaladuj(impl):
    m = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16,
                                             attn_implementation=impl).to('cuda').eval()
    return m

def jeden_przebieg(m):
    ids = torch.tensor([IDS], device='cuda')
    with torch.no_grad():
        lp = torch.log_softmax(m(ids).logits[0].float(), dim=-1)
    return [float(lp[t, IDS[t + 1]]) for t in range(len(IDS) - 1)]

m = zaladuj('eager')
proba = jeden_przebieg(m)
print('pozycji:', len(proba), '| pierwsze 3:', [round(x, 4) for x in proba[:3]])
assert len(proba) == 228, 'zla liczba pozycji'
assert all(x < 0 for x in proba), 'logprob musi byc ujemny — cos jest nie tak'
print('OK — dopiero teraz pelny pomiar')

In [ ]:
# 5. Pelny pomiar: 2 implementacje uwagi x 3 przebiegi
import time

wyniki = {}
for impl in ('eager', 'sdpa'):
    mm = m if impl == 'eager' else zaladuj(impl)
    t0 = time.monotonic()
    przebiegi = [jeden_przebieg(mm) for _ in range(3)]
    wyniki[impl] = przebiegi
    print(f'{impl}: 3 przebiegi w {time.monotonic() - t0:.1f}s')
    if impl != 'eager':
        del mm; torch.cuda.empty_cache()

for impl, przebiegi in wyniki.items():
    plik = KAT / f'd2_A_rocm_{impl}.json'
    json.dump({'ramie': 'A-ROCm', 'opis': f'transformers {impl}, batch=1, fp16, ROCm',
               'model': MODEL, 'tokenow': len(IDS), 'odcisk_input_ids': ODCISK_IDS,
               'srodowisko': srodowisko, 'przebiegi': przebiegi}, open(plik, 'w'))
    print('zapisano', plik)

## 6. Werdykt

Dwa **rozne** pytania, liczone osobno:

1. **Powtarzalnosc wewnetrzna** — czy AMD powtarza sam siebie? Jesli nie,
   problemem jest niedeterminizm, a nie vendor.
2. **Zgodnosc z CUDA** — czy verifier na NVIDIA odtworzy wynik node'a na AMD?

Wynik "kazda strona stabilna, ale AMD != NVIDIA" jest **najwazniejszy**:
oznacza deterministyczna roznice implementacji, ktorej zadna tolerancja
na szum nie zalatwi.

In [ ]:
# Baza CUDA (Franko, RTX 3080 Ti) wbudowana w notebook — bez uploadu
import base64, zlib, math, statistics

BAZA_B64 = 'eNrtmM1uJMcRhF+F4Jk7rr/MrDLgg3Tz0WdBECiJ0hL2LhckBQE2/O7+orqrZ3jyC7QALcDunqr8iYiMqv/cvz5+eX66/+vd/Xf3D3f3L9+e3/TH++vj17ffXl6/PL2+Pdw9Pf7+9Ppw9/Pj+y+f/5Yf7n77ll1ff3n59elf+vwffz59/Yv+KRf7lC/2/ae/f317f/3jl3d99v7yz6evL3/yYSmDv5+/fvvj/afnX99++vz49pnH5sMr/3nvPWp2Dza5//b676efn59+f+aLH374NC6pVBt5DCt1FOsPd5/sEqPm5rnWVkq3wrN0yZmPeurdLbrZGPNpzbl7M2u9l9SG+fy9tdZb8KylWvXzchnNo5RUk9fhvOVhvoxcRuVLc9YtFjyMSy+1j2YpRS3svwVkVV+5J8JqdW6dso+SUkk91Z4GCfLYL31kxd5rypWHPGsXEsnVkhEnSWz5BJlZi2LefESJfc2cUrORGoG5ebXtcathyZ38I5PkthPx8eNRiCjYcfuSONmBlcl09DpsBmDZSipjKLZZOWIqNbWUUqdYo2xFatSweScaqt3zXsxOFUiejZPPwtVLtkplQ92NmU/mt6xoOdi+RrRZYDb2Dg6oRm1jhshvE3XRSkYBIm/FzGxNkWtKuQ/KMZd0G+wT1rvFqlCUTF2jjATsYuhhv3Q6RluBWOupxdayACY0gf7sWxNjSkQ3Wi25tew2y9hAVHPL3jIxxfyQDtYELLuPaq3YDLylIMcAlzR3awIPG3Vm1z6A5NwGrFU2Hjmyh9PCLR4WS8PIhcxT32DF1oIYAWUapEqMSxstEV6mNCBh7uFeKCxg8RQbIkm5jZ6LpXCQMjaYmEcNla+nEnW2QNjR41IyHSCi3Lc6EkbOMXrPREVQMx4omiCHVfdeZmfyhZIVqwGTshC4rclDETtIGoKlPPL2e4tWvcYsedq+pagZIPIpDEl5MjSnS1H8LqLVvNcXmfBWvECRBgwWdax2BxrUDBTOD423dLyQfnKqND/sVSBJkyhAbF8yD5jnwq7HzJyOGSWGyFAtgGvMltFRNqaPfM8ytpcDbSHETt6t7uUsJAhtx6DvZFT2gkiUaA5hjgaH9o9BZJVkwfDg8x1cBEolQGFHPsauJokwk5EiG6TYCyVC2+iQGTlCT/MuhpVn5J9Dcob87J0uRA6goAFqE3sMMJ5kUSOYKlRt/S/wFK6KVxC8t6UeTrnpCNTOsaecoUulWQhWFuNXyuJxiDX0llbUFTHMhNvoaXEpTGwbtkJZjOo4pRu+9M5UrkUfqr8HB7dJo6o/G7hUeWpZESdXNVDKhe/CT4tE0wBQ9vWUCtL+guTyfhWZDPheYwW1QiBXTyE8IIM7ELLP9jMdUqcnCAVbQtptYYhNecjaC7VfGSM6XWpvGmSkk9aLRgJwCVFuQdb7fhlVQ3it9AEyvO3FQIYy8wQ6EzJC19YiepKTVe0LbtemEMekykREqG2VCcXLA1ZaYXX4OVWuwF5EXdIOc+LpU9pHCzXudBb4V8JaBRkpxhZE1GsgSEUiO/IGYoj4+hqyRgBgcF+E61k+JANxJIgOxunFdcusggFg1s6l+3pMd1qDE442dUS+7kAjGbRK4jUgGNw+8mfC82Fugg+RxnpBLokCEEkpdWcHmwYkJRzmHVq0r+IF9ecb+pSchhzYzviRTg7RqVfepIP3NicAP4JoNxl1jS9jYCP5CNEKkdpSVOhPn5gZZTmHyoBuSCWPNIiWAmAlTCgO6EcKvhbvLEO1KD9qWtfi1BdNB1gVyNaVfAToEYYGqealWuCS3uNiKDxTd88TrqMAOCDUEbJuUkg3u2N9xGB5oK16UJlWiTagtNQDESWwSuQ4HYo2P15UjVoKTV80d+0Qh0SCefotfGG6Ju9oUmfOoQMo0+bFgCG4Ao8mDboWHKwQOkDGwnWwf8M3U6Axvd84xlWXbwsE3ySti4bMRjRJ0is5X74uCTXYASk34Y+29qSurIqaJoxYWnwDabXKCdIaMbJPqOCOsCJIHrEJLreh41cNHgNDQJMO2GZ5jiIz7FuJ5kLDwol5uISs367DZATMNIZS+FUW5sCF9qQP7lJZrZIkw040R+v0g0WIMo+oHO5NzNmsV5f4oVqm6feh8PIxgFBOuvI6lTWnwD2jJkt0y6HZxNepWy4au32apsEAly4g8IAA7fhQG7QSpqMKQmU/Sp+1hHAuP1iPXBnlMrlqYzlm2z4swzUMs5qwCXrg3iV2hKEUbmspV4kgERXjtu9nhiKXm2R7DAG6DRP/S0LIMfOBRK50gMB0AqgqGFY65MQ0fRt8o0K+hDMVotEiwEomdLOPLDtEGX6EOEhlFrqJUU2Hcag2fuWYxBngMTR1gipSxIMOGvkUiNIx9q7CqbMAyepEBTvLMcYkJUOyRAoi3eIJM4MAEaBQH67ATzI3oFbTdlznksu/4SIpUUnbUavICpAjk4ZI/VY7+dA0bdJU4HEkSytw3NRepVseBwdQKBARGjuj++04OBFkNK1BQLGCcZKiYK7jkM4IB3BwioThUJapuBRBrowasDQJh/5dMU6/1CQr2JC9kmDf8B0EaOqX7d6Fcx6DTQ6ebI8dmYGIpIlX/DFPOI1DLW5Gh0+mHeK6IIYfnSccVsbmaKbcUoRyTRazq6GZFtfxWaQ19FaeaJ1PEK0hp00JcMwfmCxXj45x9pKxWmKZ9SKLiSbb7nEDYgqJ6NIxslhwwq03284xTLp8pXIQUhKhqJvfTAZpusIdOMsSB0eY+vOADGgyPm07ufJZ5eyKngVHhnYbPiOqafRQOjHliDLUQRVUPnx2Nl8ElCQ9ZIZhoz4ImitwCN0Bn8TnymV6A9JcAzHXw+ljN6R1OubLGR7p6mDe5wmPsZe2UzZDV6OEEoG2iJ3LfqkCPWhQJZjGN3Ys5KnpifSuHlQrWojT3STc8A0kIIfFITM9wkd/6K08OKONNoEA1OuQCtN0hFUiaMvlWjaOcQgsigASGCdXsRA3GVgydlUHg4mrst27MFOQWZvIPbgFlYUGKo3zrXsHEFydJGUWY2AWboPV6UrDDO0vAOAAnAiASeeFGJP6lQIEpEsX/CmZ9qP5oGbgKZI8mm+jrzCEiFAraUV8yU2ouCHOpjpvM/4ZN9sRfEZZZXgqgyzfjlygNs8UdB+i7946ZOYZV9hUwPsBW2PeF2DQQ9dEi2J4WPR4HoGhpR/jw3LZ7E7I8pYrZWSTs4wblcM3TsHOOpg1DNM8gjPt683GfIS24emqHFS/zkuGEzoeOvZ7P+ZEcTasPl8KmfvjXgWxpv912bU+loiJqpgd5l+9WjI3rJvkgbBudMN0HQIpBXU5uaXYhA9EkT/Z7a0GKjBh0HRAtF1/GdZdQNfRkWN/7T8+3J0Xh+fF4XlxeF4cnheH58XheXF4XhyeF4fnxeF5cXheHJ4Xh+fF4XlxeF4cnheH58XheXF4XhyeF4fnxeF5cXheHJ4Xh+fF4XlxeF4cnheH58XheXF4XhyeF4fnxeF5cXheHJ4Xh+fF4XlxeF4c/r+Lwx//+z9GMiN4'
baza = json.loads(zlib.decompress(base64.b64decode(BAZA_B64)))
print('baza CUDA:', baza['opis'], '|', baza['tokenow'], 'tokenow,',
      len(baza['przebiegi']), 'przebiegi')

def porownaj(a, b):
    d = [abs(x - y) for x, y in zip(a, b) if not (math.isnan(x) or math.isnan(y))]
    if not d: return None
    ds = sorted(d)
    return {'n': len(d), 'bit_ident_%': 100 * sum(1 for x in d if x == 0.0) / len(d),
            'mediana': statistics.median(d), 'p99': ds[min(len(ds) - 1, int(0.99 * len(ds)))],
            'maks': max(d)}

def linia(nazwa, s):
    print(f"{nazwa:<34}{s['bit_ident_%']:>9.2f}%{s['mediana']:>13.3e}{s['maks']:>13.3e}")

print('\n' + '=' * 70)
print('1) POWTARZALNOSC WEWNETRZNA')
print('=' * 70)
print(f"{'':<34}{'bit-ident':>10}{'mediana':>13}{'maks':>13}")
for impl, pr in wyniki.items():
    for i, p in enumerate(pr[1:], 2):
        linia(f'ROCm {impl}: przebieg 1 vs {i}', porownaj(pr[0], p))
for i, p in enumerate(baza['przebiegi'][1:], 2):
    linia(f'CUDA (baza): przebieg 1 vs {i}', porownaj(baza['przebiegi'][0], p))

print('\n' + '=' * 70)
print('2) ZGODNOSC MIEDZY VENDORAMI  (ROCm vs CUDA)')
print('=' * 70)
print(f"{'':<34}{'bit-ident':>10}{'mediana':>13}{'maks':>13}")
for impl, pr in wyniki.items():
    linia(f'ROCm {impl}  vs  CUDA', porownaj(pr[0], baza['przebiegi'][0]))

print('\nCzytanie wyniku:')
print('  bit-ident 100%          -> odcisk przenosi sie miedzy vendorami')
print('  wewnatrz stabilne, ale')
print('  miedzy vendorami rozne  -> DETERMINISTYCZNA roznica implementacji')
print('  wewnatrz niestabilne    -> najpierw niedeterminizm, potem vendor')

In [ ]:
# 7. Pakiet do zabrania (pobierz z panelu plikow Jupytera)
paczka = KAT / 'd2_rocm_wyniki.json'
json.dump({'srodowisko': srodowisko, 'odcisk_input_ids': ODCISK_IDS,
           'rocm': wyniki, 'baza_cuda_opis': baza['opis']}, open(paczka, 'w'))
print('pobierz:', paczka)
print('\nPAMIETAJ: "Turn-off Session" — bezczynny pod zjada limit 3 h.')